---

_You are currently looking at **version 1.1** of this notebook. To download notebooks and datafiles, as well as get help on Jupyter notebooks in the Coursera platform, visit the [Jupyter Notebook FAQ](https://www.coursera.org/learn/python-text-mining/resources/d9pwm) course resource._

---

# Assignment 3

In this assignment you will explore text message data and create models to predict if a message is spam or not. 

In [1]:
import pandas as pd
import numpy as np

spam_data = pd.read_csv('spam.csv')

spam_data['target'] = np.where(spam_data['target']=='spam',1,0)
spam_data.head(10)

,text,target
0,"Go until jurong point, crazy.. Available only ...",0
1,Ok lar... Joking wif u oni...,0
2,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,U dun say so early hor... U c already then say...,0
4,"Nah I don't think he goes to usf, he lives aro...",0
5,FreeMsg Hey there darling it's been 3 week's n...,1
6,Even my brother is not like to speak with me. ...,0
7,As per your request 'Melle Melle (Oru Minnamin...,0
8,WINNER!! As a valued network customer you have...,1
9,Had your mobile 11 months or more? U R entitle...,1


In [2]:
from sklearn.model_selection import train_test_split


X_train, X_test, y_train, y_test = train_test_split(spam_data['text'], 
                                                    spam_data['target'], 
                                                    random_state=0)

### Question 1
What percentage of the documents in `spam_data` are spam?

*This function should return a float, the percent value (i.e. $ratio * 100$).*

In [3]:
def answer_one():
    target = spam_data.groupby('target').agg('count')
    return target.iloc[1]['text'] * 100 / (target.iloc[0]['text'] + target.iloc[1]['text']) #Your answer here

In [4]:
answer_one()

13.406317300789663

### Question 2

Fit the training data `X_train` using a Count Vectorizer with default parameters.

What is the longest token in the vocabulary?

*This function should return a string.*

In [5]:
from sklearn.feature_extraction.text import CountVectorizer

def answer_two():
    vect = CountVectorizer().fit(X_train) 
    list = vect.get_feature_names()
    
    return sorted(list, key=len)[-1] #Your answer here

In [8]:
answer_two()

'com1win150ppmx3age16subscription'

### Question 3

Fit and transform the training data `X_train` using a Count Vectorizer with default parameters.

Next, fit a fit a multinomial Naive Bayes classifier model with smoothing `alpha=0.1`. Find the area under the curve (AUC) score using the transformed test data.

*This function should return the AUC score as a float.*

In [9]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import roc_auc_score

def answer_three():
    vect = CountVectorizer().fit(X_train) 
    X_train_vectorized = vect.transform(X_train)
    
    model = MultinomialNB(alpha=0.1)
    model.fit(X_train_vectorized, y_train)
    
    predictions = model.predict(vect.transform(X_test))
    auc = roc_auc_score(y_test, predictions)
    
    return auc #Your answer here

In [10]:
answer_three()

0.97208121827411165

### Question 4

Fit and transform the training data `X_train` using a Tfidf Vectorizer with default parameters.

What 20 features have the smallest tf-idf and what 20 have the largest tf-idf?

Put these features in a two series where each series is sorted by tf-idf value and then alphabetically by feature name. The index of the series should be the feature name, and the data should be the tf-idf.

The series of 20 features with smallest tf-idfs should be sorted smallest tfidf first, the list of 20 features with largest tf-idfs should be sorted largest first. 

*This function should return a tuple of two series
`(smallest tf-idfs series, largest tf-idfs series)`.*

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

def answer_four():    
    vect = TfidfVectorizer().fit(X_train) 
    X_train_vectorized = vect.transform(X_train)
    
    feature_names = np.array(vect.get_feature_names())
    sorted_tfidf_index = X_train_vectorized.max(0).toarray()[0].argsort()
    
    smallest_tfidf = pd.DataFrame({'name': feature_names[sorted_tfidf_index[:20]], 'data': sorted_tfidf_index[:20]})
    smallest_tfidf = smallest_tfidf.sort_values(['data', 'name'])
    smallest_tfidf = smallest_tfidf.set_index('name')
    smallest_tfidf_sorted = pd.Series(smallest_tfidf['data'], index=smallest_tfidf.index)

    largest_tfidf = pd.DataFrame({'name': feature_names[sorted_tfidf_index[:-21:-1]], 'data':sorted_tfidf_index[:-21:-1]})
    largest_tfidf = largest_tfidf.sort_values(['data', 'name'], ascending=False)
    largest_tfidf = largest_tfidf.set_index('name')
    largest_tfidf_sorted = pd.Series(largest_tfidf['data'], index=largest_tfidf.index)
    
    return (smallest_tfidf_sorted, largest_tfidf_sorted) #Your answer here


In [12]:
answer_four()

(name
 aaniye           689
 athletic        1046
 chef            1676
 companion       1833
 courageous      1948
 dependable      2150
 determined      2172
 diwali          2259
 exterminator    2586
 healer          3196
 listener        3919
 mornings        4332
 organizer       4715
 pest            4887
 psychiatrist    5176
 psychologist    5178
 pudunga         5184
 stylist         6197
 sympathetic     6305
 venaam          6877
 Name: data, dtype: int64, name
 yup          7328
 where        7089
 too          6598
 tick         6517
 thanx        6439
 thank        6434
 okie         4651
 ok           4646
 nite         4523
 lei          3848
 home         3293
 havent       3180
 er           2496
 done         2298
 blank        1300
 beerage      1214
 anytime       933
 anything      931
 645           533
 146tf150p     274
 Name: data, dtype: int64)

### Question 5

Fit and transform the training data `X_train` using a Tfidf Vectorizer ignoring terms that have a document frequency strictly lower than **3**.

Then fit a multinomial Naive Bayes classifier model with smoothing `alpha=0.1` and compute the area under the curve (AUC) score using the transformed test data.

*This function should return the AUC score as a float.*

In [13]:
def answer_five():
    vect = TfidfVectorizer(min_df=3).fit(X_train) 
    X_train_vectorized = vect.transform(X_train)
    
    model = MultinomialNB(alpha=0.1)
    model.fit(X_train_vectorized, y_train)
    
    predictions = model.predict(vect.transform(X_test))
    auc = roc_auc_score(y_test, predictions)     
    
    return auc #Your answer here

In [14]:
answer_five()

0.94162436548223349

### Question 6

What is the average length of documents (number of characters) for not spam and spam documents?

*This function should return a tuple (average length not spam, average length spam).*

In [15]:
def answer_six():
    spam_data['length'] = spam_data['text'].apply(lambda x: len(x))
    target = spam_data.groupby('target').agg({'length': 'mean'})
    
    return (target.iloc[0]['length'], target.iloc[1]['length']) #Your answer here

In [16]:
answer_six()

(71.023626943005183, 138.8661311914324)

<br>
<br>
The following function has been provided to help you combine new features into the training data:

In [17]:
def add_feature(X, feature_to_add):
    """
    Returns sparse feature matrix with added feature.
    feature_to_add can also be a list of features.
    """
    from scipy.sparse import csr_matrix, hstack
    return hstack([X, csr_matrix(feature_to_add).T], 'csr')

### Question 7

Fit and transform the training data X_train using a Tfidf Vectorizer ignoring terms that have a document frequency strictly lower than **5**.

Using this document-term matrix and an additional feature, **the length of document (number of characters)**, fit a Support Vector Classification model with regularization `C=10000`. Then compute the area under the curve (AUC) score using the transformed test data.

*This function should return the AUC score as a float.*

In [18]:
from sklearn.svm import SVC

def answer_seven():
    vect = TfidfVectorizer(min_df=5).fit(X_train) 
    X_train_vectorized = vect.transform(X_train)
    X_train_length = X_train.apply(lambda x: len(x))
    X_train_add_feature = add_feature(X_train_vectorized, X_train_length)
    
    model = SVC(C=10000)
    model.fit(X_train_add_feature, y_train)
    
    X_test_vectorized = vect.transform(X_test)
    X_test_length = X_test.apply(lambda x: len(x))
    X_test_add_feature = add_feature(X_test_vectorized, X_test_length)
    predictions = model.predict(X_test_add_feature)
    auc = roc_auc_score(y_test, predictions)     
    
    return auc #Your answer here

In [19]:
answer_seven()

0.95813668234215565

### Question 8

What is the average number of digits per document for not spam and spam documents?

*This function should return a tuple (average # digits not spam, average # digits spam).*

In [20]:
def answer_eight():
    spam_data['digits'] = spam_data['text'].str.findall(r'\d')
    spam_data['digits'] = spam_data['digits'].apply(lambda x: len(x))
    target = spam_data.groupby('target').agg({'digits': 'mean'})
    
    return (target.iloc[0]['digits'], target.iloc[1]['digits']) #Your answer here

In [21]:
answer_eight()

(0.29927461139896372, 15.759036144578314)

### Question 9

Fit and transform the training data `X_train` using a Tfidf Vectorizer ignoring terms that have a document frequency strictly lower than **5** and using **word n-grams from n=1 to n=3** (unigrams, bigrams, and trigrams).

Using this document-term matrix and the following additional features:
* the length of document (number of characters)
* **number of digits per document**

fit a Logistic Regression model with regularization `C=100`. Then compute the area under the curve (AUC) score using the transformed test data.

*This function should return the AUC score as a float.*

In [22]:
from sklearn.linear_model import LogisticRegression

def answer_nine():
    vect = TfidfVectorizer(min_df=5, ngram_range=(1,3)).fit(X_train) 
    X_train_vectorized = vect.transform(X_train)
    X_train_length = X_train.apply(lambda x: len(x))
    X_train_digits = X_train.str.findall(r'\d').apply(lambda x: len(x))
    X_train_add_features = add_feature(X_train_vectorized, [X_train_length, X_train_digits])
    
    model = LogisticRegression(C=100)
    model.fit(X_train_add_features, y_train)
    
    X_test_vectorized = vect.transform(X_test)
    X_test_length = X_test.apply(lambda x: len(x))
    X_test_digits = X_test.str.findall(r'\d').apply(lambda x: len(x))
    X_test_add_features = add_feature(X_test_vectorized, [X_test_length, X_test_digits])
    predictions = model.predict(X_test_add_features)
    auc = roc_auc_score(y_test, predictions)     
    
    return auc #Your answer here

In [23]:
answer_nine()

0.96533283533945646

### Question 10

What is the average number of non-word characters (anything other than a letter, digit or underscore) per document for not spam and spam documents?

*Hint: Use `\w` and `\W` character classes*

*This function should return a tuple (average # non-word characters not spam, average # non-word characters spam).*

In [24]:
def answer_ten():
    spam_data['non_word'] = spam_data['text'].str.findall(r'\W').apply(lambda x: len(x))
    target = spam_data.groupby('target').agg({'non_word': 'mean'})
    
    return (target.iloc[0]['non_word'], target.iloc[1]['non_word']) #Your answer here

In [25]:
answer_ten()

(17.291813471502589, 29.041499330655956)

### Question 11

Fit and transform the training data X_train using a Count Vectorizer ignoring terms that have a document frequency strictly lower than **5** and using **character n-grams from n=2 to n=5.**

To tell Count Vectorizer to use character n-grams pass in `analyzer='char_wb'` which creates character n-grams only from text inside word boundaries. This should make the model more robust to spelling mistakes.

Using this document-term matrix and the following additional features:
* the length of document (number of characters)
* number of digits per document
* **number of non-word characters (anything other than a letter, digit or underscore.)**

fit a Logistic Regression model with regularization C=100. Then compute the area under the curve (AUC) score using the transformed test data.

Also **find the 10 smallest and 10 largest coefficients from the model** and return them along with the AUC score in a tuple.

The list of 10 smallest coefficients should be sorted smallest first, the list of 10 largest coefficients should be sorted largest first.

The three features that were added to the document term matrix should have the following names should they appear in the list of coefficients:
['length_of_doc', 'digit_count', 'non_word_char_count']

*This function should return a tuple `(AUC score as a float, smallest coefs list, largest coefs list)`.*

In [26]:
def answer_eleven():
    vect = CountVectorizer(min_df=5, ngram_range=(2,5), analyzer='char_wb').fit(X_train) 
    X_train_vectorized = vect.transform(X_train)
    X_train_length = X_train.apply(lambda x: len(x))
    X_train_digits = X_train.str.findall(r'\d').apply(lambda x: len(x))
    X_train_non_word = X_train.str.findall(r'\W').apply(lambda x: len(x))
    X_train_add_features = add_feature(X_train_vectorized, [X_train_length, X_train_digits, X_train_non_word])
    
    model = LogisticRegression(C=100)
    model.fit(X_train_add_features, y_train)
    
    X_test_vectorized = vect.transform(X_test)
    X_test_length = X_test.apply(lambda x: len(x))
    X_test_digits = X_test.str.findall(r'\d').apply(lambda x: len(x))
    X_test_non_word = X_test.str.findall(r'\W').apply(lambda x: len(x))
    X_test_add_features = add_feature(X_test_vectorized, [X_test_length, X_test_digits, X_test_non_word])
    predictions = model.predict(X_test_add_features)
    auc = roc_auc_score(y_test, predictions)     
    
    feature_names = np.array(vect.get_feature_names())
    feature_names = np.append(feature_names, ['length_of_doc', 'digit_count', 'non_word_char_count'])
    sorted_coef_index = model.coef_[0].argsort()

    smallest_coefficients = pd.Series(sorted_coef_index[:10], index=feature_names[sorted_coef_index[:10]])
    smallest_coefficients_sorted = smallest_coefficients.sort_values()
    
    largest_coefficients = pd.Series(sorted_coef_index[:-11:-1], index=feature_names[sorted_coef_index[:-11:-1]])
    largest_coefficients_sorted = largest_coefficients.sort_values(ascending=False)
    
    return (auc, smallest_coefficients_sorted, largest_coefficients_sorted) #Your answer here

In [27]:
answer_eleven()

(0.97885931107074342,  go    1143
  h     1189
  i     1310
  m     1620
  y     2907
 .      3218
 ..     3227
 :)     4471
 ?      4520
 go     8245
 dtype: int64, digit_count    16315
 xt             15949
 ww             15855
 ne             10911
 mob            10589
 ia              8804
 co              6160
 ar              5158
  x              2893
  ch              658
 dtype: int64)